In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin 
import pandas as pd
import re
import requests
import time
import cloudscraper
import json
import asyncio
from playwright.async_api import async_playwright
import numpy as np
import openpyxl

In [3]:
# Leilao Zuk

async def extrair_dataframe_zuk():
    base_url = "https://www.portalzuk.com.br"
    lista_lotes = []

    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        page = await browser.new_page()
        await page.goto("https://www.portalzuk.com.br/leilao-de-veiculos", wait_until="networkidle")

        # Clica em "carregar mais" até sumir
        # Clica em "carregar mais" até sumir ou ficar desabilitado
        while True:
            try:
                # Fecha o modal se estiver aberto
                modal = page.locator("#modalVirada")
                if await modal.count() > 0 and await modal.is_visible():
                    # Tenta clicar no botão de fechar do modal
                    close_btn = modal.locator("button[data-dismiss], button.close, .btn-close")
                    if await close_btn.count() > 0:
                        await close_btn.first.click()
                    else:
                        # Força fechar via JavaScript
                        await page.evaluate("document.getElementById('modalVirada').style.display='none'")
                    await page.wait_for_timeout(500)

                btn = page.locator("#btn_carregarMais")
                if await btn.count() == 0:
                    print("botão sumiu do DOM")
                    break
                if not await btn.is_visible():
                    print("botão invisível")
                    break
                if await btn.is_disabled():
                    print("botão disabled")
                    break

                # Clica via JavaScript para ignorar sobreposições
                await page.evaluate("document.getElementById('btn_carregarMais').click()")
                await page.wait_for_timeout(2000)
            except Exception as e:
                print(f"erro no loop: {e}")
                break

        html = await page.content()
        await browser.close()

    soup = BeautifulSoup(html, 'html.parser')
    cards = soup.find_all('div', class_='card-property')

    for card in cards:
        link_tag = card.find('a', href=True)
        if not link_tag:
            continue
        url_lote = urljoin(base_url, link_tag.get('href'))

        try:
            address_span = card.find('address').find('span', title=True)
        except:
            address_span = None
        detalhe_completo = address_span['title'] if address_span else ""
        partes_detalhe = [p.strip() for p in detalhe_completo.split(',')]
        marca_modelo = "N/A"
        ano_ano = "N/A"
        if len(partes_detalhe) > 1:
            marca_modelo = partes_detalhe[1]
            busca_ano = re.search(r'(\d{4}/\d{4})', detalhe_completo)
            if busca_ano:
                ano_ano = busca_ano.group(1)
        marca = marca_modelo.split('/')[0] if '/' in marca_modelo else marca_modelo
        modelo = marca_modelo.split('/')[1] if '/' in marca_modelo else "N/A"

        price_items = card.find_all('li', class_='card-property-price')
        valor_venda = "N/A"
        data_leilao = "N/A"

        for item in price_items:
            val_span = item.find('span', class_='card-property-price-value')
            date_span = item.find('span', class_='card-property-price-data')
            if val_span:
                for child in val_span.find_all('span'):
                    child.decompose()
                texto_v = val_span.get_text(strip=True)
                match = re.search(r'[\d.]+,\d+', texto_v)
                valor_venda = match.group(0) if match else "N/A"
            if date_span:
                data_leilao = date_span.get_text(strip=True)

        loc_span = card.find('span', style=lambda x: x and 'margin-left:2.5rem' in x)
        cidade_uf = loc_span.get_text(strip=True) if loc_span else "N/A"

        lista_lotes.append({
            "marca": marca,
            "modelo": modelo,
            "ano/ano": ano_ano,
            "valor": valor_venda,
            "cidade_uf": cidade_uf,
            "link": url_lote,
            "site": "Portal Zuk",
            "data_do_leilao": data_leilao
        })

    df = pd.DataFrame(lista_lotes)
    df['valor'] = (
    df['valor']
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
    return df

df_zuk = await extrair_dataframe_zuk()

df_zuk = df_zuk[df_zuk['cidade_uf'].str.contains('Curitiba', case=False, na=False)].reset_index(drop=True)
df_zuk

botão invisível


,marca,modelo,ano/ano,valor,cidade_uf,link,site,data_do_leilao


In [4]:
# Leilao Favareto

# Configurações iniciais
url_root = "https://www.favaretoleiloes.com.br"
url_leiloes_disponiveis = f"{url_root}/leiloes/"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}

def extrair_dados_veiculos(soup_leilao, data_texto):
    lotes_leilao = []
    linhas = soup_leilao.find_all('tr', class_=['lista-lotes-par', 'lista-lotes-impar'])
    
    for linha in linhas:
        try:
            # 1. Descrição e Ano
            td_obs = linha.find('td', class_='obs')
            if not td_obs: continue
            descricao_bruta = td_obs.get_text(strip=True)
            
            # Busca Ano/Ano (Ex: 2015/2016)
            busca_ano = re.search(r'(\d{4}/\d{4})', descricao_bruta)
            ano_ano = busca_ano.group(1) if busca_ano else "N/A"
            
            # 2. Marca e Modelo
            # Pega o que vem antes da primeira vírgula e tenta separar pela barra
            # 1. Remove o prefixo 'I/' e limpa espaços
            parte_identificacao = descricao_bruta.replace('I/', '').strip()

            # 2. Pega apenas o que vem antes da primeira vírgula (remove ano/detalhes extras)
            parte_identificacao = parte_identificacao.split(',')[0].strip()

            if '/' in parte_identificacao:
                # Divide pela barra
                split_barra = parte_identificacao.split('/')
                marca = split_barra[0].strip().upper()
                
                # O modelo é o que vem após a barra (já garantimos que não tem vírgula acima)
                modelo = split_barra[1].strip().upper()
            else:
                marca = parte_identificacao.upper()
                modelo = "N/A"

            # 3. Valor (Lance Atual)
            td_valor = linha.find('td', id=lambda x: x and x.startswith('VL'))
            valor_str = td_valor.get_text(strip=True) if td_valor else "0"
            valor_num = 0.0
            if any(c.isdigit() for c in valor_str):
                valor_limpo = valor_str.replace('R$', '').replace('.', '').replace(',', '.').strip()
                valor_num = float(valor_limpo)

            # 4. Situação e Estado
            td_situacao = linha.find('td', id=lambda x: x and x.startswith('ST'))
            situacao = td_situacao.get_text(strip=True).upper() if td_situacao else "N/A"
            td_estado = linha.find('td', class_='centro-estado')
            estado = td_estado.get_text(strip=True).upper() if td_estado else "NORMAL"

            # 5. Link do Lote (extraído do onclick do JS)
            link_tag = linha.find('a', class_='btn-lote')
            link_final = "N/A"
            km_valor = "N/A" # Default caso não encontre

            if link_tag and 'onclick' in link_tag.attrs:
                ids = re.findall(r'\d+', link_tag['onclick'])
                if len(ids) >= 2:
                    link_final = f"{url_root}/lance/{ids[0]}/{ids[1]}/"
                    leilao_id = ids[0] # Ex: 1603
                    lote_seq = ids[1]  # Ex: 1
                    
                    # URL do endpoint que o site usa para carregar os dados
                    url_json = "https://www.favaretoleiloes.com.br/classes/json_lance.php"
                    
                    # Dados que o site espera no POST
                    payload = {
                        'ordem': lote_seq,
                        'leilao': leilao_id
                    }

                    try:
                        # Fazemos um POST em vez de GET, imitando o site
                        response_json = requests.post(url_json, data=payload, headers=headers, timeout=10)
                        dados_lote = response_json.json() # Transforma a resposta em dicionário Python
                        
                        # Agora pegamos o KM direto da fonte
                        km_valor = dados_lote.get('km', 'N/A')
                        
                    except Exception as e:
                        print(f"❌ Erro ao capturar JSON: {e}")
                        km_valor = "Erro no JSON"

            lotes_leilao.append({
                'marca': marca,
                'modelo': modelo,
                'ano/ano': ano_ano,
                'km': km_valor, # Agora com o valor real
                'valor': valor_num,
                'situacao': f"{situacao} | {estado}",
                'link': link_final,
                'site': "Favareto Leilões",
                'data_do_leilao': data_texto
            })
        except Exception as e:
            continue
            
    return lotes_leilao

# --- ETAPA 1: Requisição Principal ---
try:
    print(f"🔍 Acessando: {url_leiloes_disponiveis}")
    res_principal = requests.get(url_leiloes_disponiveis, headers=headers)
    soup_principal = BeautifulSoup(res_principal.text, 'html.parser')
    
    
    # Encontra todos os botões "Visitar Lotes"
    links_visitar = soup_principal.find_all('a', string=re.compile(r'visitar lotes', re.IGNORECASE))
    
    lista_final_geral = []

    # --- ETAPA 2: Requisições de cada Leilão ---
    for link in links_visitar:
        href = link.get('href')
        url_leilao = f"{url_root}/{href.lstrip('/')}"
        
        print(f"  📂 Entrando no leilão: {url_leilao}")
        res_leilao = requests.get(url_leilao, headers=headers)
        soup_leilao = BeautifulSoup(res_leilao.text, 'html.parser')
        
        # Captura a data deste leilão específico (fica no topo da página de lotes)
        data_tag = soup_leilao.find('div', id='dados')
        data_texto = data_tag.find('h3').get_text(strip=True) if data_tag and data_tag.find('h3') else "N/A"
        
        # --- ETAPA 3: Tratamento e Extração ---
        dados_extraidos = extrair_dados_veiculos(soup_leilao, data_texto)
        lista_final_geral.extend(dados_extraidos)
        
        time.sleep(1) # Pausa para evitar bloqueio

    # Geração do DataFrame Final
    df_favareto = pd.DataFrame(lista_final_geral)
    
    # Filtro opcional: Apenas Curitiba (Favareto é muito forte aqui)
    if not df_favareto.empty:
        # Muitas vezes a localização está na descrição ou na situação
        print(f"✅ Total de {len(df_favareto)} lotes extraídos do Favareto.")
    else:
        print("⚠️ Nenhum lote encontrado.")

except Exception as e:
    print(f"🚨 Erro geral: {e}")

df_favareto 

🔍 Acessando: https://www.favaretoleiloes.com.br/leiloes/
  📂 Entrando no leilão: https://www.favaretoleiloes.com.br/edital/1608
  📂 Entrando no leilão: https://www.favaretoleiloes.com.br/edital/1609
  📂 Entrando no leilão: https://www.favaretoleiloes.com.br/edital/1610
  📂 Entrando no leilão: https://www.favaretoleiloes.com.br/edital/1611
  📂 Entrando no leilão: https://www.favaretoleiloes.com.br/edital/1612
✅ Total de 114 lotes extraídos do Favareto.


,marca,modelo,ano/ano,km,valor,situacao,link,site,data_do_leilao
0,FIAT,UNO WAY 1.0,2010/2011,199849,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1608/1/,Favareto Leilões,31/03/2026 - Terça-Feira
1,CITROEN,C4 20EXCA5P F,2010/2011,138498,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1608/2/,Favareto Leilões,31/03/2026 - Terça-Feira
2,VW,GOL 1.0,2011/2012,169566,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1608/3/,Favareto Leilões,31/03/2026 - Terça-Feira
3,CHEVROLET,CELTA 1.0L LS,2013/2013,128392,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1608/4/,Favareto Leilões,31/03/2026 - Terça-Feira
4,VW,GOL CL MB,2015/2016,153221,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1608/5/,Favareto Leilões,31/03/2026 - Terça-Feira
...,...,...,...,...,...,...,...,...,...
109,CITROEN,C4L THP A ORIG,2017/2018,172348,0.0,NÃO LOTEADO | SUCATA,https://www.favaretoleiloes.com.br/lance/1609/52/,Favareto Leilões,01/04/2026 - Quarta-Feira
110,VW,VOYAGE 1.6L MB5,2019/2020,52888,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1609/53/,Favareto Leilões,01/04/2026 - Quarta-Feira
111,HYUNDAHB20 10M EVOLUTI,N/A,2021/2022,141211,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1609/54/,Favareto Leilões,01/04/2026 - Quarta-Feira
112,VW,NIVUS HL TSI,2022/2022,10855,0.0,NÃO LOTEADO |,https://www.favaretoleiloes.com.br/lance/1609/55/,Favareto Leilões,01/04/2026 - Quarta-Feira


In [5]:
# Leilao Sodre

def renovar_token(scraper):
    # 1. Acessa a página principal
    response = scraper.get("https://www.sodresantoro.com.br/veiculos/lotes")
    
    # 2. Procura o padrão apiSanctumToken:"..." dentro do HTML usando Regex
    token_match = re.search(r'apiSanctumToken:"([^"]+)"', response.text)
    
    if token_match:
        novo_token = token_match.group(1)
        return novo_token
    else:
        print("Não consegui encontrar o token no HTML.")
        return None


url = "https://www.sodresantoro.com.br/api/search-lots"
scraper = cloudscraper.create_scraper()

token_atualizado = renovar_token(scraper)

# Payload com os seus filtros originais
payload_dict = {
    "indices": ["veiculos", "judiciais-veiculos"],
    "query": {
        "bool": {
            "filter": [
                {"bool": {"should": [{"bool": {"must": [{"term": {"auction_status": "online"}}]}}, {"bool": {"must": [{"term": {"auction_status": "aberto"}}], "must_not": [{"terms": {"lot_status_id": [5, 7]}}]}}, {"bool": {"must": [{"term": {"auction_status": "encerrado"}}, {"terms": {"lot_status_id": [6]}}]}}], "minimum_should_match": 1}},
                {"bool": {"should": [{"bool": {"must_not": {"term": {"lot_status_id": 6}}}}, {"bool": {"must":[{"term": {"lot_status_id": 6}}, {"term": {"segment_id": 1}}]}}], "minimum_should_match": 1}},
                {"bool": {"should": [{"bool": {"must_not": [{"term": {"lot_test": True}}]}}], "minimum_should_match": 1}}
            ]
        }
    },
    "post_filter": {
        "bool": {
            "filter": [
                {"terms": {"lot_origin": ["seguro 0 km", "financiamento"]}},
                {"terms": {"lot_location": ["curitiba i/pr", "curitiba ii/pr"]}},
                {"terms": {"lot_category": ["carros"]}}
            ]
        }
    },
    "from": 0,
    "size": 48,
    "sort": [{"lot_status_id_order": {"order": "asc"}}, {"bid_actual": {"order": "asc"}}]
}


headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Content-Type': 'application/json',
    'x-api-sanctum-token': 'UfyZp1J3WBC2Y7Qwo7y0o034rBX0FwhKzrF4T9lX',
    'Origin': 'https://www.sodresantoro.com.br',
    'Referer': 'https://www.sodresantoro.com.br/veiculos/lotes'
}

if token_atualizado:
    headers['x-api-sanctum-token'] = token_atualizado

# Garantindo cookies
scraper.get("https://www.sodresantoro.com.br/")

response = scraper.post(url, data=json.dumps(payload_dict), headers=headers)

if response.status_code == 200:
    data = response.json()

    lotes_brutos = data.get('results', [])
    
    lista_final = []
    for lote in lotes_brutos:
        # 1. IDs e Link
        auction_id = lote.get('auction_id')
        lot_id = lote.get('lot_id')
        link_leilao = f"https://leilao.sodresantoro.com.br/leilao/{auction_id}/lote/{lot_id}/"


        data_init = lote.get('auction_date_init', '')
        if data_init and len(data_init) >= 16:
            data_parte, hora_parte = data_init.split(' ')
            ano, mes, dia = data_parte.split('-')
            data_formatada = f"{dia}/{mes}/{ano}"
            hora_formatada = hora_parte[:5] + "h"
            data_final = f"{data_formatada} - {hora_formatada}"
        else:
            data_final = "N/A"

        # 3. Situação (Origem + Status)
        situacao = f"{lote.get('lot_origin', '')} - {lote.get('lot_status', '')}".strip().upper()

        # 4. Montagem padronizada
        lista_final.append({
            'marca': str(lote.get('lot_brand', '')).upper(),
            'modelo': str(lote.get('lot_model', '')).upper(),
            'ano/ano': f"{lote.get('lot_year_manufacture')}/{lote.get('lot_year_model')}",
            'km': lote.get('lot_km', 0),
            'valor': float(lote.get('bid_actual', 0.0)),
            'situacao': situacao,
            'link': link_leilao,
            'site': "Sodré Santoro",
            'data_do_leilao': data_final 
        })

    df_sodre = pd.DataFrame(lista_final)

else:
    print(f"Erro {response.status_code}")

df_sodre.head()

,marca,modelo,ano/ano,km,valor,situacao,link,site,data_do_leilao
0,FIAT,SIENA ELX FLEX,2006/2007,180985,7000.0,FINANCIAMENTO - ANDAMENTO,https://leilao.sodresantoro.com.br/leilao/2824...,Sodré Santoro,26/03/2026 - 09:30h
1,CHEVROLET,CLASSIC LS,2010/2011,0,7000.0,FINANCIAMENTO - ANDAMENTO,https://leilao.sodresantoro.com.br/leilao/2824...,Sodré Santoro,26/03/2026 - 09:30h
2,RENAULT,DUSTER 20 D 4X2A,2011/2012,126756,12000.0,FINANCIAMENTO - ANDAMENTO,https://leilao.sodresantoro.com.br/leilao/2824...,Sodré Santoro,26/03/2026 - 09:30h
3,VOLKSWAGEN,GOL 1.0,2010/2011,120115,12500.0,FINANCIAMENTO - ANDAMENTO,https://leilao.sodresantoro.com.br/leilao/2824...,Sodré Santoro,26/03/2026 - 09:30h
4,FORD,KA SE 1.0 HA B,2017/2018,150767,14000.0,FINANCIAMENTO - ANDAMENTO,https://leilao.sodresantoro.com.br/leilao/2824...,Sodré Santoro,26/03/2026 - 09:30h


In [6]:
# Leilao LoopLeiloes

url = "https://api.loopleiloes.com.br/search/leilao"

# Passando uma lista no dicionário de parâmetros
params = {
    "localizacao": ["Curitiba-PR", "São José Dos Pinhais-PR"],
    "tipo": "Leve",
    "numero_veiculos": "300" # Aumentei para pegar de ambos os locais
}

try:
    response = requests.get(url, params=params, headers=headers)
    response.raise_for_status()  # Verifica se a requisição deu certo
    
    dados = response.json()

    veiculos_processados = []
    veiculos = dados.get('hits', [])
    
    for item in veiculos:
        source = item.get('_source', {})
        
        # 1. Tratamento da Data do Leilão
        # No JSON enviado, source['event']['date'] e source['estimatedStartTime'] estão None
        # Vamos tratar para não quebrar o código
        data_bruta = source.get('estimatedStartTime')
        if data_bruta:
            # Caso venha no padrão ISO: 2026-02-25T14:00:00
            try:
                data_parte, hora_parte = data_bruta.split('T')
                ano_l, mes_l, dia_l = data_parte.split('-')
                data_final = f"{dia_l}/{mes_l}/{ano_l} - {hora_parte[:5]}h"
            except:
                data_final = data_bruta
        else:
            data_final = "Aguardando Leilão"

        # 2. Montagem no seu padrão de colunas
        veiculo = {
            'marca': str(source.get('brand', '')).upper(),
            'modelo': str(source.get('model', '')).upper(),
            'ano/ano': f"{source.get('manufactureYear')}/{source.get('modelYear')}",
            'km': source.get('mileage', 0),
            'valor': source.get('lastBid') if source.get('lastBid') else 0.0,
            'situacao': "ATIVO" if not source.get('lotStatus') else str(source.get('lotStatus')).upper(),
            'link': f"https://www.loopleiloes.com.br/veiculo/{source.get('slug')}",
            'site': "Loop Leilões",
            'data_do_leilao': data_final
        }
        veiculos_processados.append(veiculo)
    
    df_loop = pd.DataFrame(veiculos_processados)

except Exception as e:
    print(f"Erro ao acessar a API: {e}")

pd.set_option('display.max_colwidth', None)
df_loop

,marca,modelo,ano/ano,km,valor,situacao,link,site,data_do_leilao
0,CITROEN,C4,2010/2011,144628,10000.0,RETIRADO,https://www.loopleiloes.com.br/veiculo/4-l-5l-2010-2011-60095782,Loop Leilões,2026-03-26 11:21:40
1,CITROEN,C4,2007/2008,206055,9000.0,ABERTO_PARA_LANCE,https://www.loopleiloes.com.br/veiculo/4-ll-20-2007-2008-60096157,Loop Leilões,2026-03-26 11:45:00
2,CHEVROLET,ONIX,2020/2021,219249,30000.0,ABERTO_PARA_LANCE,https://www.loopleiloes.com.br/veiculo/l-10-l-2-2020-2021-60097488,Loop Leilões,2026-03-26 12:06:40
3,VOLKSWAGEN,UP,2019/2020,1,30000.0,ABERTO_PARA_LANCE,https://www.loopleiloes.com.br/veiculo/l-2019-2020-60097504,Loop Leilões,2026-03-26 11:58:20
4,FORD,ECOSPORT,2010/2011,106653,15000.0,ABERTO_PARA_LANCE,https://www.loopleiloes.com.br/veiculo/l1-6-l-2010-2011-60096735,Loop Leilões,2026-03-26 11:35:50
...,...,...,...,...,...,...,...,...,...
230,RENAULT,KWID,2020/2021,66779,0.0,ATIVO,https://www.loopleiloes.com.br/veiculo/renault-kwid-1-0-12v-sce-flex-zen-manual-2020-2021-60094495,Loop Leilões,Aguardando Leilão
231,MITSUBISHI,L200 TRITON,2020/2020,184469,0.0,ATIVO,https://www.loopleiloes.com.br/veiculo/mitsubishi-l200-triton-2-4-16v-turbo-diesel-sport-gls-cd-4p-4x4-automatico-2020-2020-60096405,Loop Leilões,Aguardando Leilão
232,VOLKSWAGEN,JETTA,2011/2012,178904,0.0,ATIVO,https://www.loopleiloes.com.br/veiculo/volkswagen-jetta-2-0-tsi-highline-200cv-gasolina-4p-tiptronic-2011-2012-60096594,Loop Leilões,Aguardando Leilão
233,RENAULT,KWID,2024/2025,11280,0.0,ATIVO,https://www.loopleiloes.com.br/veiculo/renault-kwid-1-0-12v-sce-flex-outsider-manual-2024-2025-60097095,Loop Leilões,Aguardando Leilão


In [ ]:
# Leilao Superbid

url = "https://www.superbid.net/categorias/carros-motos/carros/parana/curitiba-pr?searchType=opened&filter=auction.subMarketplaces.subMarketplaceDesc:judicial&pageNumber=1&pageSize=30"

headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers)
    response.encoding = 'utf-8'
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    
except Exception as e:
    print(f"Erro: {e}")


script_tag = soup.find('script', id='__NEXT_DATA__')

if script_tag:
    data = json.loads(script_tag.string)
    
    # Caminho comum no Next.js da Superbid
    offers = data['props']['pageProps']['offersList']['offers']
    
    lista_lotes = []
    # 'offers' 
    for car in offers:
        product = car.get('product', {})
        auction = car.get('auction', {})
        
        # 1. Tratamento da Data do Leilão (Pegando endDate do topo do lote)
        data_bruta = car.get('endDate', '')  # Ex: '2026-03-10 18:03:00'
        if data_bruta and len(data_bruta) >= 16:
            try:
                data_p, hora_p = data_bruta.split(' ')
                ano_s, mes_s, dia_s = data_p.split('-')
                data_final = f"{dia_s}/{mes_s}/{ano_s} - {hora_p[:5]}h"
            except:
                data_final = data_bruta
        else:
            data_final = "Consultar Site"

        # 2. Extração de Marca e Modelo (Busca em múltiplos lugares)
        marca = product.get('brand', {}).get('description')
        modelo = product.get('model', {}).get('description')
        
        # Se vier vazio, tenta buscar no template de identificação
        if not marca or not modelo:
            template_groups = product.get('template', {}).get('groups', [])
            for group in template_groups:
                if group.get('id') == 'identificacao':
                    for prop in group.get('properties', []):
                        if prop.get('id') == 'marca': marca = prop.get('value')
                        if prop.get('id') == 'modelo': modelo = prop.get('value')

        # 3. Ano/Ano (Tenta anofabricacao/anomodelo no template)
        ano_fab = "N/A"
        ano_mod = "N/A"
        template_groups = product.get('template', {}).get('groups', [])
        for group in template_groups:
            for prop in group.get('properties', []):
                if prop.get('id') == 'anofabricacao': ano_fab = prop.get('value')
                if prop.get('id') == 'anomodelo': ano_mod = prop.get('value')
        
        # Se não achou no template, tenta o yearModel geral
        if ano_fab == "N/A":
            ano_val = product.get('yearModel', 'N/A')
            ano_fab, ano_mod = ano_val, ano_val

        # 4. KM (Busca na descrição se o campo mileage for 0)
        km = product.get('mileage', 0)
        if km == 0:
            # Tenta extrair da descrição: "está com 228669km rodados"
            desc_text = product.get('detailedDescription', '')
            match_km = re.search(r'(\d+)km', desc_text.replace('.', '').replace(' ', ''))
            if match_km:
                km = match_km.group(1)

        # Montagem padronizada
        lista_lotes.append({
            'marca': str(marca).upper() if marca else "N/A",
            'modelo': str(modelo).upper() if modelo else "N/A",
            'ano/ano': f"{ano_fab}/{ano_mod}",
            'km': km,
            'valor': float(car.get('price', 0.0)),
            'situacao': f"JUDICIAL - {auction.get('judicialPracaDescription', 'ATIVO')}".upper(),
            'link': f"https://www.superbid.net/oferta/{car.get('id')}",
            'site': "Superbid",
            'data_do_leilao': data_final
        })
    
df_superbid = pd.DataFrame(lista_lotes)
df_superbid = df_superbid[~(df_superbid['marca'] == 'N/A')]

In [8]:
# Leilao Copart
async def run_copart():
    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True) # Coloquei True para ser mais rápido
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        print("🚀 Acessando Copart Curitiba...")
        # URL com seus filtros: Uso Normal + Pátio Curitiba
        url = "https://www.copart.com.br/search/leil%C3%A3o/?displayStr=Leil%C3%A3o&from=%2FvehicleFinder&searchCriteria=%7B%22query%22:%5B%22*%22%5D,%22filter%22:%7B%22MISC%22:%5B%22tipovenda:Leil%C3%A3o%22%5D,%22patioleilao%22:%5B%22patioleilao:%5C%22Curitiba%20-%20PR%5C%22%22%5D%7D,%22sort%22:%5B%22auction_date_utc%20asc%22,%22brazil_default_sort%20asc%22%5D,%22watchListOnly%22:false,%22searchName%22:%22%22,%22freeFormSearch%22:false,%22itemsPerPage%22:100%7D"
        
        await page.goto(url, wait_until="networkidle")

        table_selector = "#serverSideDataTable"
        await page.wait_for_selector(f"{table_selector} tbody tr")

        rows = await page.query_selector_all(f"{table_selector} tbody tr")
        
        lista_final = []
        for row in rows:
            cells = await row.query_selector_all("td")
            if len(cells) > 10:
                # 1. Marca e Modelo (Células 4 e 5 no novo layout)
                marca = (await cells[4].inner_text()).strip().upper()
                modelo = (await cells[5].inner_text()).strip().upper()

                # 2. Ano
                ano_val = (await cells[3].inner_text()).strip()

                # 3. Valor (Lance Atual) - Busca pelo data-uname para evitar erro de índice
                valor_cell = await row.query_selector('[data-uname="lotsearchLothighbid"]')
                valor_texto = await valor_cell.inner_text() if valor_cell else "0"
                
                # Limpeza robusta: remove R$, BRL e espaços, troca vírgula por ponto
                # Ex: "R$ 8.050,00 BRL" -> "8050.00"
                valor_limpo = re.sub(r'[^\d,]', '', valor_texto).replace(',', '.')
                try:
                    valor = float(valor_limpo)
                except:
                    valor = 0.0

                # 4. Data do Leilão
                data_cell = await row.query_selector('[data-uname="lotsearchLotauctiondate"]')
                data_final = (await data_cell.inner_text()).replace('\n', ' ').strip() if data_cell else "N/A"
                
                condicao_raw = await cells[8].inner_text()
                condicao = condicao_raw.replace('\n', ' ').strip()

                # 5. Link (Ajustado para o seu HTML que está na cells[1])
                link_tag = await cells[1].query_selector("a")
                link_relative = await link_tag.get_attribute("href") if link_tag else ""
                link_lote = "https://www.copart.com.br" + link_relative.replace('.', '', 1)

                lista_final.append({
                    'marca': marca,
                    'modelo': modelo,
                    'ano/ano': f"{ano_val}/{ano_val}",
                    'valor': valor,
                    'link': link_lote,
                    'situacao': condicao,
                    'site': "Copart",
                    'data_do_leilao': data_final
                })

        await browser.close()
        return pd.DataFrame(lista_final)

# Execução
df_copart = await run_copart()

🚀 Acessando Copart Curitiba...


In [9]:
# Leilao bradesco

url = "https://api.vitrinebradesco.com.br/v1/auctions?page=1&type=vehicles&ufs=PR&vehicle_type_of_recovery=resumed"
headers = {'User-Agent': 'Mozilla/5.0'}

try:
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    json_data = response.json()

    lista_veiculos = json_data.get('data', [])
    lista_final = []

    for item in lista_veiculos:
        # 1. Extração de Marca, Modelo e Anos do campo 'name'
        # Exemplo: 'HONDA - CITY EX FLEX - 2011 / 2011'
        nome_bruto = item.get('name', '')
        
        # Separamos por '-' para tentar isolar Marca e Modelo
        partes = [p.strip() for p in nome_bruto.split('-')]
        marca = partes[0].upper() if len(partes) > 0 else "N/A"
        modelo = partes[1].upper() if len(partes) > 1 else "N/A"
        
        # Buscamos o padrão de anos (ex: 2011 / 2011) usando Regex
        busca_anos = re.findall(r'(\d{4})', nome_bruto)
        if len(busca_anos) >= 2:
            ano_ano = f"{busca_anos[-2]}/{busca_anos[-1]}"
        elif len(busca_anos) == 1:
            ano_ano = f"{busca_anos[0]}/{busca_anos[0]}"
        else:
            ano_ano = "N/A"

        # 2. Tratamento da Data (Formato ISO: 2026-02-27T14:00:00.000Z)
        data_iso = item.get('auction_date', '')
        if data_iso and 'T' in data_iso:
            data_p, hora_p = data_iso.split('T')
            ano, mes, dia = data_p.split('-')
            data_final = f"{dia}/{mes}/{ano} - {hora_p[:5]}h"
        else:
            data_final = "N/A"

        # 3. Montagem padronizada
        lista_final.append({
            'marca': marca,
            'modelo': modelo,
            'ano/ano': ano_ano,
            'km': 0,  # A API Vitrine geralmente não traz KM na listagem principal
            'valor': float(item.get('price', 0.0)),
            'situacao': "Recuperado financiamento",
            'link': item.get('auctioneer', {}).get('website', ''), # Link do leiloeiro
            'site': "Vitrine Bradesco",
            'data_do_leilao': data_final
        })

    df_bradesco = pd.DataFrame(lista_final)


except Exception as e:
    print(f"Erro ao gerar DataFrame: {e}")

# Exibe o resultado
pd.set_option('display.max_colwidth', None)
print(df_bradesco)

  marca               modelo    ano/ano  km     valor  \
0  FIAT  PALIO WK ADVEN DUAL  2012/2012   0   13700.0   
1  FORD       FOCUS SE 1.6 H  2014/2014   0   13100.0   
2  FIAT       UNO VIVACE 1.0  2015/2016   0   11700.0   
3   MAN     TGX 29.480 6X4 T  2019/2020   0  112300.0   

                   situacao                            link              site  \
0  Recuperado financiamento  https://www.vipleiloes.com.br/  Vitrine Bradesco   
1  Recuperado financiamento  https://www.vipleiloes.com.br/  Vitrine Bradesco   
2  Recuperado financiamento  https://www.vipleiloes.com.br/  Vitrine Bradesco   
3  Recuperado financiamento  https://www.vipleiloes.com.br/  Vitrine Bradesco   

        data_do_leilao  
0  30/03/2026 - 14:00h  
1  30/03/2026 - 14:00h  
2  30/03/2026 - 14:00h  
3  30/03/2026 - 19:00h  


In [10]:
# Leilao Claudio Kuss
import asyncio
import random
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

async def run_claudio_kuss():
    url_base = "https://www.claudiokussleiloes.com.br"
    url_proximos = f"{url_base}/proximos-leiloes"
    api_url = f"{url_base}/json_edital.php"
    headers = {'User-Agent': 'Mozilla/5.0', 'X-Requested-With': 'XMLHttpRequest'}
    all_data = []

    async with async_playwright() as p:
        browser = await p.firefox.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) Firefox/120.0")
        page = await context.new_page()

        print("🚀 Acessando lista de leilões...")
        try:
            response = requests.get(url_proximos, headers=headers, timeout=20)
            soup = BeautifulSoup(response.text, 'html.parser')
            paineis = soup.find_all('div', class_='panel-body')
        except Exception as e:
            print(f"❌ Erro ao acessar lista principal: {e}")
            await browser.close()
            return pd.DataFrame()

        for painel in paineis:
            link_tag = painel.find('a', class_='btn-success')
            titulo_tag = painel.find('h2')

            if link_tag and titulo_tag:
                link_relativo = link_tag.get('href', '')
                leilao_id = link_relativo.split('/')[-1]
                nome_leilao = titulo_tag.text.strip()
                
                print(f"\n📦 Processando: {nome_leilao}")

                payload_qtd = {'op': 'Q', 'leilaoID': leilao_id, 'pesq': ''}
                try:
                    res_qtd = requests.post(api_url, headers=headers, data=payload_qtd, timeout=15)
                    total_paginas = int(res_qtd.json().get('qtdePag', 1))
                except:
                    continue 

                for pag in range(1, total_paginas + 1):
                    payload_dados = {'leilaoID': leilao_id, 'op': 'P', 'pag': str(pag), 'loteado': 'S', 'pesq': ''}
                    try:
                        res_dados = requests.post(api_url, headers=headers, data=payload_dados, timeout=15)
                        veiculos_json = res_dados.json()
                    except:
                        continue 

                    for item in veiculos_json:
                        seq_veiculo = item.get('seq')
                        lote_num = item.get('lote')
                        link_lote = f"{url_base}/lance/{leilao_id}/{seq_veiculo}"
                        
                        try:
                            await page.goto(link_lote, wait_until="domcontentloaded", timeout=20000)
                            obs_selector = 'span[name="tblInformacoes"]'
                            await page.wait_for_selector(obs_selector, timeout=4000)
                            
                            observacao = await page.locator(obs_selector).inner_text()
                            acessorios = await page.locator('span[name="tblAcessorios"]').inner_text()
                        except:
                            observacao = "OBS não carregada"
                            acessorios = ""

                        all_data.append({
                            "ID_Leilao": leilao_id, "Leilao_Nome": nome_leilao,
                            "Lote": lote_num, "Descricao": item.get('bem'),
                            "Valor_Lance": item.get('valor'), "Observacao": observacao.strip(),
                            "Acessorios": acessorios.strip(), "Ano": item.get('ano'),
                            "Link_Veiculo": link_lote
                        })
                        await page.wait_for_timeout(random.randint(500, 1000)) # Delay levemente reduzido
                    
                    print(f"   ✅ Página {pag}/{total_paginas} concluída.")

        await browser.close()

    # --- PROCESSAMENTO PANDAS ---
    df = pd.DataFrame(all_data)
    if not df.empty:
        # Garante que campos de texto não sejam None para não quebrar o .str
        df['Descricao'] = df['Descricao'].fillna('').astype(str)
        
        def extrair(row):
            desc_completa = (row['Descricao'] + " " + row['Observacao']).upper()
            km_match = re.search(r'(\d[\d\.]*)\s?KM', desc_completa)
            km = km_match.group(1).replace('.', '') if km_match else "0"
            
            situacao = "CIRCULÁVEL"
            if any(x in desc_completa for x in ["SUCATA", "BAIXADO"]): situacao = "SUCATA"
            elif any(x in desc_completa for x in ["MEDIA MONTA", "SINISTRO"]): situacao = "MÉDIA MONTA"
            
            return pd.Series([km, situacao])

        df[['km', 'situacao']] = df.apply(extrair, axis=1)
        
        # Criação do DF final com as regras de Marca e Modelo solicitadas
        df_claudioKuss = pd.DataFrame({
            # Remove IMP e pega antes da barra
            'marca': df['Descricao'].str.replace(r'^IMP\s+', '', regex=True).str.split('/').str[0].str.strip().str.upper(),
            
            # Pega após a barra e remove o que vem depois da vírgula
            'modelo': df['Descricao'].str.split('/').str[1].str.split(',').str[0].str.strip().str.upper().fillna('N/A'),
            
            'ano/ano': df['Ano'],
            'km': df['km'],
            
            # Conversão numérica segura (o que falhar vira NaN, não quebra o código)
            'valor': pd.to_numeric(
                df['Valor_Lance'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), 
                errors='coerce'
            ),
            
            'situacao': df['situacao'],
            'observacao': df['Observacao'],
            'link': df['Link_Veiculo'],
            'site': 'Claudio Kuss Leilões',
            'data_do_leilao': df['Leilao_Nome']
        })
        return df_claudioKuss
        
    return pd.DataFrame()

# Execução no Jupyter
df_claudioKuss = await run_claudio_kuss()
df_claudioKuss

🚀 Acessando lista de leilões...

📦 Processando: Curitiba : 31/03/2026 - 10:00 horas
   ✅ Página 1/26 concluída.
   ✅ Página 2/26 concluída.
   ✅ Página 3/26 concluída.
   ✅ Página 4/26 concluída.
   ✅ Página 5/26 concluída.
   ✅ Página 6/26 concluída.
   ✅ Página 7/26 concluída.
   ✅ Página 8/26 concluída.
   ✅ Página 9/26 concluída.
   ✅ Página 10/26 concluída.
   ✅ Página 11/26 concluída.
   ✅ Página 12/26 concluída.
   ✅ Página 13/26 concluída.
   ✅ Página 14/26 concluída.
   ✅ Página 15/26 concluída.
   ✅ Página 16/26 concluída.
   ✅ Página 17/26 concluída.
   ✅ Página 18/26 concluída.
   ✅ Página 19/26 concluída.
   ✅ Página 20/26 concluída.
   ✅ Página 21/26 concluída.
   ✅ Página 22/26 concluída.
   ✅ Página 23/26 concluída.
   ✅ Página 24/26 concluída.
   ✅ Página 25/26 concluída.
   ✅ Página 26/26 concluída.

📦 Processando: Curitiba : 07/04/2026 - 10:00 horas


,marca,modelo,ano/ano,km,valor,situacao,observacao,link,site,data_do_leilao
0,CITROEN,C4 20 VTR,08/08,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
1,CHEV,TRACKER 12T A PR,22/23,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
2,FIAT,PALIO FIRE FLEX,07/07,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
3,FIAT,FIORINO FLEX FURGAO,10/11,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
4,VW,AMAROK CD 4X4 S AB C DUPLA,12/12,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
...,...,...,...,...,...,...,...,...,...,...
247,FORD,FIESTA FLEX,10/11,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
248,PEUGEOT,307 SD 16 PR PK,08/08,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
249,HYUNDAI,HB20 1.6M 1.6M,13/13,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas
250,FIAT,TORO FREEDOM AT AB C DUPLA,16/17,0,NaN,CIRCULÁVEL,,https://www.claudiokussleiloes.com.br/lance/865/0,Claudio Kuss Leilões,Curitiba : 31/03/2026 - 10:00 horas


In [11]:
# Junção de todos os leilões

def normalizar_valor(v):
    s = str(v).strip()
    
    # Já é numérico limpo (ex: 21211.55)
    if re.match(r'^\d+(\.\d+)?$', s):
        return float(s)
    
    # Formato BR: 21.211,55
    if ',' in s and '.' in s:
        return float(s.replace('.', '').replace(',', '.'))
    
    # Só vírgula: 21211,55
    if ',' in s and '.' not in s:
        return float(s.replace(',', '.'))
    
    # Só ponto: pode ser milhar (21.211) ou decimal (21.55)
    # Se mais de 3 casas após o ponto, é milhar
    if '.' in s:
        partes = s.split('.')
        if len(partes[-1]) > 3:
            return float(s.replace('.', ''))
        return float(s)
    
    return float(s) if s.replace('.','').replace(',','').isdigit() else 0.0



# 1. Criamos uma lista com todos os DataFrames que você gerou
# Usamos uma lista para facilitar o tratamento de possíveis tabelas vazias
dfs = [
    df_zuk, 
    df_favareto, 
    df_sodre, 
    df_loop, 
    df_superbid, 
    df_bradesco, 
    df_copart, 
    df_claudioKuss
]



# 2. Concatenamos apenas os DataFrames que não estiverem vazios
# ignore_index=True serve para criar um novo índice de 0 até o total de linhas
df_final = pd.concat([df for df in dfs if not df.empty], ignore_index=True)

# 3. Limpeza de Dados:
# Remove possíveis anúncios duplicados (baseado no link, que é único)
df_final = df_final.drop_duplicates(subset=['link'])

# Garante que a coluna valor seja numérica para ordenação correta
df_final['valor'] = pd.to_numeric(df_final['valor'], errors='coerce').fillna(0)

# 4. Ordenação: do mais barato para o mais caro
df_final = df_final.sort_values(by='valor', ascending=True)

df_final['ano ref'] = df_final['ano/ano'].str.split('/').str[0]

# 2. Aplica a regra dos 4 caracteres
df_final['ano ref'] = df_final['ano ref'].apply(lambda x: f"20{x}" if len(str(x)) < 4 else str(x))

mask = (df_final['observacao'].notnull()) & (df_final['observacao'].str.strip() != "")

df_final['situacao'] = np.where(
    mask, 
    "Problema", 
    df_final['situacao']
)

df_final['ano ref'] = pd.to_numeric(df_final['ano ref'], errors='coerce')

df_final['km'] = pd.to_numeric(df_final['km'], errors='coerce')

# Visualização
marcas_banidas = ['IVECO','DAF', 'SR', 'MEPEL', 'CAMC', 'TANGJUN', 'NEW HOLLAND', 'TATU', 'VENCE TUDO', 'CASE', 'SOCMA', 'PICCIN', 'MAXXFORTE', 'OUTROS', 'JOHN DEERE','JACTO','JINBEI','JTZ','KF','LIFAN','LR','MMC','NANJING','RETIRADO','SHINERAY','YAMAHA']


# Criando a máscara de filtro
filtro = (
    ~df_final['marca'].isin(marcas_banidas)
) & (
    ~df_final['situacao'].str.upper().str.contains('SINIS', na=False)
) & (
    ~df_final['situacao'].str.upper().str.contains('SUC', na=False)
) & (
    ~df_final['situacao'].str.upper().str.contains('COLI', na=False)
) & (
    ~df_final['situacao'].str.upper().str.contains('ENCH', na=False)
)



# Aplicando o filtro
df_filtrado = df_final[filtro]

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


def calcular_relevancia(row):
    pontos = 0
    
    # --- Regra 1: Ano ref ---
    ano = row['ano ref']
    if pd.notna(ano):
        if ano > 2019:
            pontos += 2
        elif ano > 2014:
            pontos += 1
    
    # --- Regra 2: KM ---
    km = row['km']
    if pd.notna(km):
        if 1 < km < 60000:
            pontos += 2
        elif km < 100000:
            pontos += 1
    
    # --- Regra 3: Valor ---
    valor = row['valor']
    if pd.notna(valor):
        if 40000 <= valor <= 70000:
            pontos += 2
        elif valor > 70000:
            pontos += 1
        elif valor > 0:
            pontos += 0.5
    
    return pontos

df_filtrado = df_filtrado.copy()
df_filtrado['relevancia'] = df_filtrado.apply(calcular_relevancia, axis=1)

# Opcional: ver o resultado ordenado por relevância decrescente
df_filtrado.sort_values('relevancia', ascending=False)


df_filtrado.to_excel("radar_leiloes_curitiba.xlsx", index=False)

